# 1. vllm 라이브러리 설치

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
!uv pip install --system vllm==0.11.0 torch==2.8.0 transformers==4.57.6

# 2. 모델 로드

## 2-1. Exaone 모델 로드

In [ ]:
!vllm serve LGAI-EXAONE/EXAONE-4.0-1.2B \
--max_model_len 2048 \
--gpu-memory-utilization 0.45 \
--port 8000 > exaone_server.log 2>&1 &

In [ ]:
!cat exaone_server.log

## 2-2. Qwen 모델 로드

In [ ]:
!vllm serve Qwen/Qwen3-0.6B \
--max_model_len 2048 \
--gpu-memory-utilization 0.45 \
--port 8001 > qwen_server.log 2>&1 &

In [ ]:
cat qwen_server.log

## 2-3. 프로세스별 GPU 사용량 확인

In [ ]:
!nvidia-smi

# 3. 모델 추론

In [ ]:
messages = [
    {"role": "user", "content": "서울의 경복궁은 어떤 곳이야?"}
]

In [ ]:
payload = {
    "messages": messages,
    "temperature": 0.7
}

headers={"Content-Type": "application/json"}

In [ ]:
eaxaone_endpoint = "http://localhost:8000/v1/chat/completions"
qwen_endpoint = "http://localhost:8001/v1/chat/completions"

## 3-1. Exaone 모델 요청 및 응답

In [ ]:
# Exaone 모델 요청
import httpx

async with httpx.AsyncClient(timeout=30.0) as client:
    response = await client.post(url=eaxaone_endpoint, json=payload, headers=headers)

In [ ]:
# Exaone 모델 응답
import json

if response.status_code == 200:
    model_response = response.json()
    print(json.dumps(model_response, indent=4, ensure_ascii=False))

else:
    print(f"에러 발생: {response.status_code}")
    print(response.text)

## 3-2. Qwen 모델 요청 및 응답

In [ ]:
# Qwen 모델

async with httpx.AsyncClient(timeout=30.0) as client:
    response = await client.post(url=qwen_endpoint, json=payload, headers=headers)

In [ ]:
# Qwen 모델 응답

if response.status_code == 200:
    model_response = response.json()
    print(json.dumps(model_response, indent=4, ensure_ascii=False))

else:
    print(f"에러 발생: {response.status_code}")
    print(response.text)